In [1]:
import requests
from ipywidgets import widgets
from IPython.display import display, HTML, clear_output

# --- API Configuration ---
# Key is entered securely at runtime and never stored in the notebook
import getpass
API_KEY = getpass.getpass("Enter your Google Fact Check Tools API Key: ")

def fetch_fact_checks(query, api_key, lang="en"):
    """Queries the Google Fact Check Tools API."""
    url = "https://factchecktools.googleapis.com/v1alpha1/claims:search"
    params = {
        "query": query,
        "key": api_key,
        "languageCode": lang
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as http_err:
        return {"error": f"HTTP error occurred: {http_err} (Check if your API Key restriction allows this request style)"}
    except Exception as err:
        return {"error": f"An error occurred: {err}"}

def display_results(data, query):
    """Formats and displays the JSON output into clean HTML cards."""
    if "error" in data:
        display(HTML(f"<p style='color: red;'><b>Error:</b> {data['error']}</p>"))
        return

    if "claims" not in data or not data["claims"]:
        display(HTML(f"<div style='padding: 10px; background-color: #f8d7da; color: #721c24; border-radius: 5px;'>"
                     f"No explicit fact-check records found for: <b>'{query}'</b></div>"))
        return

    html_output = f"<h3>Fact Check Results for: <i>'{query}'</i></h3>"

    for idx, claim_item in enumerate(data["claims"], 1):
        claim_text = claim_item.get("text", "N/A")
        claimant = claim_item.get("claimant", "Unknown Source")
        claim_date = claim_item.get("claimDate", "Unknown date")[:10]

        html_output += f"""
        <div style="border: 1px solid #ddd; padding: 15px; margin-bottom: 15px; border-radius: 8px; background-color: #fafafa; font-family: sans-serif;">
            <p style="font-size: 16px; margin: 0 0 8px 0;"><b>[{idx}] Claim:</b> "{claim_text}"</p>
            <p style="font-size: 13px; color: #666; margin: 0 0 12px 0;">🗣️ <i>Stated by {claimant} on {claim_date}</i></p>
        """

        for review in claim_item.get("claimReview", []):
            publisher = review.get("publisher", {}).get("name", "Unknown Publisher")
            rating = review.get("textualRating", "No Rating Provided").upper()
            review_url = review.get("url", "#")
            title = review.get("title", "View Review Article")

            badge_color = "#6c757d"
            if any(word in rating for word in ["FALSE", "FAKE", "INCORRECT", "MISLEADING"]):
                badge_color = "#dc3545"
            elif any(word in rating for word in ["TRUE", "CORRECT", "ACCURATE"]):
                badge_color = "#28a745"
            elif "PART" in rating or "MIXTURE" in rating:
                badge_color = "#ffc107"

            html_output += f"""
            <div style="margin-left: 20px; padding: 10px; border-left: 4px solid {badge_color}; background-color: #fff; margin-top: 8px;">
                <p style="margin: 0 0 5px 0;"><b>Reviewer:</b> {publisher}</p>
                <p style="margin: 0 0 5px 0;"><b>Verdict:</b> <span style="background-color: {badge_color}; color: white; padding: 2px 6px; border-radius: 3px; font-size: 12px; font-weight: bold;">{rating}</span></p>
                <p style="margin: 0;"><a href="{review_url}" target="_blank" style="color: #007bff; text-decoration: none; font-weight: 500;">🔗 {title}</a></p>
            </div>
            """
        html_output += "</div>"

    display(HTML(html_output))

# --- Simple Interactive UI ---
input_text = widgets.Text(
    value='moon landing is fake',
    placeholder='Type a claim or keyword...',
    description='Claim:',
    disabled=False,
    layout=widgets.Layout(width='60%')
)

button = widgets.Button(
    description='Check Claim',
    button_style='primary',
    icon='search'
)

output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        if not input_text.value.strip():
            print("Please enter a claim text to look up.")
            return

        results = fetch_fact_checks(input_text.value, API_KEY)
        display_results(results, input_text.value)

button.on_click(on_button_clicked)

display(HTML("<h2>Google Fact Check Tools API Quick Checker</h2>"))
display(widgets.HBox([input_text, button]))
display(output)

Output()